# Colab A100 — 5-Method Full Paper Experiment (550 Runs)

이 노트북 하나로 로컬 실험과 동일한 **데이터셋·모델·5개 방법·5개 seed·epoch·FP16·effective batch 64** 조건을 Google Colab A100에서 실행합니다.

## 중요

- Colab 런타임을 반드시 **A100 GPU**로 선택합니다.
- 결과와 체크포인트는 Google Drive의 `MyDrive/paper_finetuning_5method_A100`에 저장됩니다.
- 연결이 끊기면 같은 Study 셀을 다시 실행합니다. 완료 run은 건너뛰고 중단 run은 마지막 정상 epoch checkpoint부터 재개합니다.
- 동일 Drive 폴더를 두 Colab 세션에서 동시에 실행하지 마세요.
- GPU가 다르므로 학습시간과 부동소수점 결과가 로컬과 비트 단위로 같지는 않지만, 논문 실험 프로토콜은 동일합니다.


## 1. 패키지 설치

로컬에서 검증한 Transformers·Datasets·PEFT 버전을 설치합니다. Colab의 CUDA 호환 PyTorch는 그대로 사용합니다.


In [ ]:
%pip install -q "transformers==5.9.0" "datasets==4.8.5" "peft==0.19.1" accelerate scikit-learn pandas numpy sentencepiece


## 2. Google Drive 연결 및 단일 노트북 코드 배치

다음 셀은 이 노트북에 내장된 실행 엔진과 설정을 Drive에 기록합니다. `results` 폴더는 삭제하거나 덮어쓰지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, sys

DRIVE_ROOT = Path('/content/drive/MyDrive/paper_finetuning_5method_A100')
(DRIVE_ROOT / 'src').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'config').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'src' / '__init__.py').write_text('', encoding='utf-8')
print('DRIVE_ROOT =', DRIVE_ROOT)


In [ ]:
SUITE_SOURCE = 'from __future__ import annotations\n\nimport inspect\nimport json\nimport os\nimport random\nimport shutil\nimport tempfile\nimport time\nimport traceback\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom datasets import Dataset, DatasetDict, load_dataset, load_from_disk\nfrom sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support\nfrom sklearn.model_selection import train_test_split\nfrom transformers import (\n    AutoModelForSequenceClassification,\n    AutoTokenizer,\n    DataCollatorWithPadding,\n    EarlyStoppingCallback,\n    Trainer,\n    TrainerCallback,\n    TrainingArguments,\n    set_seed,\n)\n\nROOT = Path(__file__).resolve().parents[1]\nCONFIG_PATH = ROOT / "config" / "experiment_config.json"\nMETHODS = ("full_ft", "lora", "adapter", "ia3", "bitfit")\n\n\ndef now_iso():\n    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")\n\n\ndef atomic_json(path: Path, payload):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=path.parent)\n    try:\n        with os.fdopen(fd, "w", encoding="utf-8") as handle:\n            json.dump(payload, handle, ensure_ascii=False, indent=2)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temp_name, path)\n    finally:\n        if os.path.exists(temp_name):\n            os.unlink(temp_name)\n\n\ndef append_event(path: Path, payload):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps({"time": now_iso(), **payload}, ensure_ascii=False) + "\\n")\n        handle.flush()\n\n\ndef load_config():\n    return json.loads(CONFIG_PATH.read_text(encoding="utf-8"))\n\n\ndef runtime_info():\n    return {\n        "time": now_iso(),\n        "python": os.sys.version,\n        "torch": torch.__version__,\n        "cuda_runtime": torch.version.cuda,\n        "cuda_available": torch.cuda.is_available(),\n        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",\n        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3) if torch.cuda.is_available() else 0,\n    }\n\n\ndef precheck(require_cuda=True):\n    import datasets\n    import peft\n    import transformers\n\n    info = runtime_info() | {\n        "transformers": transformers.__version__,\n        "datasets": datasets.__version__,\n        "peft": peft.__version__,\n    }\n    if require_cuda and not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU가 감지되지 않았습니다. Python (ai_lab_first) 커널인지 확인하세요.")\n    atomic_json(ROOT / "results" / "environment.json", info)\n    return info\n\n\n@dataclass(frozen=True)\nclass TaskSpec:\n    key: str\n    path: str\n    subset: str | None\n    text_col: str\n    label_col: str\n    num_labels: int\n    source_split: str | None = None\n    label_threshold: float | None = None\n    direct: str | None = None\n\n\nTASKS = {\n    "measuring_hate_speech": TaskSpec("measuring_hate_speech", "ucberkeley-dlab/measuring-hate-speech", None, "comment", "hatespeech", 2, "train", 1.0),\n    "tweet_sentiment": TaskSpec("tweet_sentiment", "cardiffnlp/tweet_eval", "sentiment", "text", "label", 3),\n    "finance_sentiment": TaskSpec("finance_sentiment", "lmassaron/FinancialPhraseBank", None, "sentence", "label", 3),\n    "movie_reviews": TaskSpec("movie_reviews", "stanfordnlp/imdb", None, "text", "label", 2),\n    "product_reviews": TaskSpec("product_reviews", "SetFit/amazon_reviews_multi_en", None, "text", "label", 5),\n    "tweet_emotion": TaskSpec("tweet_emotion", "cardiffnlp/tweet_eval", "emotion", "text", "label", 4),\n    "tweet_hate": TaskSpec("tweet_hate", "cardiffnlp/tweet_eval", "hate", "text", "label", 2),\n    "tweet_offensive": TaskSpec("tweet_offensive", "cardiffnlp/tweet_eval", "offensive", "text", "label", 2),\n    "tweet_irony": TaskSpec("tweet_irony", "cardiffnlp/tweet_eval", "irony", "text", "label", 2),\n    "news_topic": TaskSpec("news_topic", "fancyzhx/ag_news", None, "text", "label", 4),\n    "news_ynat": TaskSpec("news_ynat", "klue", "ynat", "title", "label", 7),\n    "movie_nsmc": TaskSpec("movie_nsmc", "csv", None, "document", "label", 2, direct="nsmc"),\n    "comment_kmhas_binary": TaskSpec("comment_kmhas_binary", "csv", None, "text", "label", 2, direct="kmhas"),\n}\n\n\ndef _raw_dataset(spec: TaskSpec):\n    if spec.direct == "nsmc":\n        return load_dataset("csv", data_files={\n            "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",\n            "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",\n        }, delimiter="\\t")\n    if spec.direct == "kmhas":\n        return load_dataset("csv", data_files={\n            "train": "https://raw.githubusercontent.com/adlnlp/K-MHaS/main/data/kmhas_train.txt",\n            "validation": "https://raw.githubusercontent.com/adlnlp/K-MHaS/main/data/kmhas_valid.txt",\n            "test": "https://raw.githubusercontent.com/adlnlp/K-MHaS/main/data/kmhas_test.txt",\n        }, delimiter="\\t", column_names=["text", "label"], skiprows=1)\n    return load_dataset(spec.path, spec.subset) if spec.subset else load_dataset(spec.path)\n\n\ndef _kmhas_binary(value):\n    if hasattr(value, "tolist"):\n        value = value.tolist()\n    if isinstance(value, str):\n        try:\n            value = json.loads(value)\n        except Exception:\n            value = [int(x.strip()) for x in value.split(",") if x.strip()]\n    if isinstance(value, (int, np.integer)):\n        value = [int(value)]\n    return 0 if list(value) == [8] else 1\n\n\ndef _standardize(raw, spec: TaskSpec):\n    if spec.source_split:\n        raw = DatasetDict(all=raw[spec.source_split])\n    standardized = {}\n    for split_name, split in raw.items():\n        texts, labels, ids = [], [], []\n        columns = set(split.column_names)\n        text_col = spec.text_col if spec.text_col in columns else next((x for x in ("text", "sentence", "comment", "document", "title") if x in columns), None)\n        label_col = spec.label_col if spec.label_col in columns else next((x for x in ("label", "labels", "hatespeech", "hate_speech_score") if x in columns), None)\n        if text_col is None or label_col is None:\n            raise RuntimeError(f"{spec.key}: text/label 컬럼 확인 실패: {split.column_names}")\n        for i, row in enumerate(split):\n            text = row.get(text_col)\n            raw_label = row.get(label_col)\n            if text is None or raw_label is None:\n                continue\n            if spec.key == "comment_kmhas_binary":\n                label = _kmhas_binary(raw_label)\n            elif spec.label_threshold is not None:\n                try:\n                    label = int(float(raw_label) >= spec.label_threshold)\n                except (TypeError, ValueError):\n                    continue\n            else:\n                label = int(raw_label)\n            if not 0 <= label < spec.num_labels:\n                continue\n            texts.append(str(text)); labels.append(label); ids.append(f"{spec.key}:{split_name}:{i}")\n        standardized[split_name] = Dataset.from_dict({"sample_id": ids, "text": texts, "labels": labels})\n    return DatasetDict(standardized)\n\n\ndef _split_indices(labels, test_size, seed):\n    idx = np.arange(len(labels))\n    try:\n        return train_test_split(idx, test_size=test_size, random_state=seed, stratify=np.asarray(labels))\n    except ValueError:\n        return train_test_split(idx, test_size=test_size, random_state=seed)\n\n\ndef _ensure_three_splits(ds: DatasetDict, seed=42):\n    if "all" in ds:\n        train_idx, hold_idx = _split_indices(ds["all"]["labels"], 0.2, seed)\n        hold = ds["all"].select(sorted(hold_idx.tolist()))\n        val_idx, test_idx = _split_indices(hold["labels"], 0.5, seed)\n        return DatasetDict(train=ds["all"].select(sorted(train_idx.tolist())), validation=hold.select(sorted(val_idx.tolist())), test=hold.select(sorted(test_idx.tolist())))\n    if all(x in ds for x in ("train", "validation", "test")):\n        return DatasetDict({x: ds[x] for x in ("train", "validation", "test")})\n    if "test" in ds:\n        train_idx, val_idx = _split_indices(ds["train"]["labels"], 0.1, seed)\n        return DatasetDict(train=ds["train"].select(sorted(train_idx.tolist())), validation=ds["train"].select(sorted(val_idx.tolist())), test=ds["test"])\n    if "validation" in ds:\n        train_idx, val_idx = _split_indices(ds["train"]["labels"], 0.1, seed)\n        return DatasetDict(train=ds["train"].select(sorted(train_idx.tolist())), validation=ds["train"].select(sorted(val_idx.tolist())), test=ds["validation"])\n    train_idx, hold_idx = _split_indices(ds["train"]["labels"], 0.2, seed)\n    hold = ds["train"].select(sorted(hold_idx.tolist()))\n    val_idx, test_idx = _split_indices(hold["labels"], 0.5, seed)\n    return DatasetDict(train=ds["train"].select(sorted(train_idx.tolist())), validation=hold.select(sorted(val_idx.tolist())), test=hold.select(sorted(test_idx.tolist())))\n\n\ndef _balanced_limit(ds: Dataset, limit: int | None, seed=42):\n    if not limit or len(ds) <= limit:\n        return ds\n    labels = np.asarray(ds["labels"]); idx = np.arange(len(ds))\n    chosen, _ = train_test_split(idx, train_size=limit, random_state=seed, stratify=labels)\n    return ds.select(sorted(chosen.tolist()))\n\n\ndef load_task(task_key, run_mode, limits=None):\n    if run_mode == "SMOKE":\n        cache_tag = "smoke"\n    elif limits:\n        cache_tag = "paper_" + "_".join(f"{k}{int(v)}" for k, v in sorted(limits.items()))\n    else:\n        cache_tag = "paper_full"\n    cache = ROOT / "cache" / task_key / cache_tag\n    cache_complete = (cache / "dataset_dict.json").exists() and all(\n        (cache / split / "state.json").exists() and (cache / split / "dataset_info.json").exists()\n        for split in ("train", "validation", "test")\n    )\n    if cache_complete:\n        return load_from_disk(str(cache))\n    if cache.exists():\n        broken = cache.with_name(cache.name + ".incomplete_" + datetime.now().strftime("%Y%m%d_%H%M%S"))\n        cache.rename(broken)\n    spec = TASKS[task_key]\n    ds = _ensure_three_splits(_standardize(_raw_dataset(spec), spec))\n    cfg = load_config()\n    if run_mode == "SMOKE":\n        limits = cfg["smoke_limits"]\n    if limits:\n        ds = DatasetDict({name: _balanced_limit(split, int(limits[name]), 42) for name, split in ds.items()})\n    cache.parent.mkdir(parents=True, exist_ok=True)\n    temp_cache = cache.with_name(cache.name + ".building")\n    if temp_cache.exists():\n        shutil.rmtree(temp_cache)\n    ds.save_to_disk(str(temp_cache))\n    temp_cache.rename(cache)\n    atomic_json(cache.parent / f"{cache_tag}_manifest.json", {\n        "task": task_key, "run_mode": run_mode, "source": spec.path, "subset": spec.subset,\n        "limits": limits, "rows": {k: len(v) for k, v in ds.items()}, "split_seed": 42,\n        "fingerprints": {k: getattr(v, "_fingerprint", "UNKNOWN") for k, v in ds.items()},\n    })\n    return ds\n\n\nclass BottleneckAdapter(nn.Module):\n    def __init__(self, hidden_size, bottleneck, dropout=0.0):\n        super().__init__()\n        self.down = nn.Linear(hidden_size, bottleneck)\n        self.activation = nn.GELU()\n        self.dropout = nn.Dropout(dropout)\n        self.up = nn.Linear(bottleneck, hidden_size)\n        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)\n\n    def forward(self, hidden_states):\n        return hidden_states + self.up(self.dropout(self.activation(self.down(hidden_states))))\n\n\nclass OutputWithAdapter(nn.Module):\n    def __init__(self, original, hidden_size, bottleneck, dropout):\n        super().__init__()\n        self.dense = original.dense\n        self.LayerNorm = original.LayerNorm\n        self.dropout = original.dropout\n        self.adapter = BottleneckAdapter(hidden_size, bottleneck, dropout)\n\n    def forward(self, hidden_states, input_tensor):\n        hidden_states = self.dense(hidden_states)\n        hidden_states = self.adapter(hidden_states)\n        hidden_states = self.dropout(hidden_states)\n        return self.LayerNorm(hidden_states + input_tensor)\n\n\ndef _encoder_layers(model):\n    for attr in ("roberta", "bert", "deberta", "electra"):\n        base = getattr(model, attr, None)\n        if base is not None and hasattr(base, "encoder") and hasattr(base.encoder, "layer"):\n            return base.encoder.layer\n    raise RuntimeError(f"Adapter 미지원 모델 구조: {model.__class__.__name__}")\n\n\ndef _unfreeze_head(model):\n    for name, param in model.named_parameters():\n        if any(token in name for token in ("classifier", "score")):\n            param.requires_grad = True\n\n\ndef build_model(model_name, num_labels, method, cfg):\n    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, ignore_mismatched_sizes=True, attn_implementation="eager")\n    if method == "full_ft":\n        return model\n    if method == "bitfit":\n        for param in model.parameters(): param.requires_grad = False\n        for name, param in model.named_parameters():\n            if name.endswith(".bias"): param.requires_grad = True\n        _unfreeze_head(model)\n        return model\n    if method == "adapter":\n        for param in model.parameters(): param.requires_grad = False\n        acfg = cfg["adapter"]\n        for layer in _encoder_layers(model):\n            layer.output = OutputWithAdapter(layer.output, model.config.hidden_size, acfg["bottleneck"], acfg["dropout"])\n        _unfreeze_head(model)\n        return model\n    from peft import IA3Config, LoraConfig, TaskType, get_peft_model\n    if method == "lora":\n        lcfg = cfg["lora"]\n        peft_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=lcfg["r"], lora_alpha=lcfg["alpha"], lora_dropout=lcfg["dropout"], target_modules=["query", "value"], modules_to_save=["classifier"], bias="none")\n    elif method == "ia3":\n        peft_cfg = IA3Config(task_type=TaskType.SEQ_CLS, target_modules=["key", "value", "intermediate.dense"], feedforward_modules=["intermediate.dense"], modules_to_save=["classifier"])\n    else:\n        raise ValueError(method)\n    return get_peft_model(model, peft_cfg)\n\n\ndef parameter_counts(model):\n    total = sum(p.numel() for p in model.parameters())\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    return {"trainable_params": trainable, "total_params": total, "trainable_parameter_ratio": trainable / total}\n\n\ndef compute_metrics(prediction):\n    labels = prediction.label_ids\n    preds = np.argmax(prediction.predictions, axis=-1)\n    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)\n    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1, "macro_precision": precision, "macro_recall": recall}\n\n\nclass AtomicEpochCallback(TrainerCallback):\n    def __init__(self, path):\n        self.path = Path(path)\n        if self.path.exists():\n            try:\n                self.rows = pd.read_csv(self.path).to_dict("records")\n            except Exception:\n                self.rows = []\n        else:\n            self.rows = []\n\n    def _save(self):\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        temp = self.path.with_suffix(".tmp")\n        pd.DataFrame(self.rows).to_csv(temp, index=False, encoding="utf-8-sig")\n        os.replace(temp, self.path)\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if logs:\n            self.rows.append({"time": now_iso(), "event": "log", "epoch": state.epoch, "global_step": state.global_step, **logs})\n            self._save()\n\n    def on_save(self, args, state, control, **kwargs):\n        self.rows.append({"time": now_iso(), "event": "checkpoint", "epoch": state.epoch, "global_step": state.global_step})\n        self._save()\n\n\ndef training_args(run_dir, method, seed, epochs, run_mode, cfg):\n    kwargs = dict(\n        output_dir=str(run_dir / "checkpoints"),\n        learning_rate=cfg["learning_rates"][method],\n        per_device_train_batch_size=cfg["batch_size"],\n        per_device_eval_batch_size=cfg["eval_batch_size"],\n        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],\n        num_train_epochs=1 if run_mode == "SMOKE" else epochs,\n        weight_decay=cfg["weight_decay"], warmup_ratio=cfg["warmup_ratio"],\n        logging_strategy="steps", logging_steps=20,\n        save_strategy="epoch", load_best_model_at_end=True,\n        metric_for_best_model="macro_f1", greater_is_better=True,\n        save_total_limit=1, report_to="none",\n        seed=seed, data_seed=42, fp16=cfg["precision"] == "fp16",\n        dataloader_num_workers=cfg["dataloader_num_workers"],\n    )\n    signature = inspect.signature(TrainingArguments.__init__)\n    kwargs["eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"] = "epoch"\n    return TrainingArguments(**kwargs)\n\n\ndef _latest_checkpoint(run_dir):\n    root = run_dir / "checkpoints"\n    candidates = sorted(\n        (p for p in root.glob("checkpoint-*") if (p / "trainer_state.json").exists()),\n        key=lambda p: int(p.name.split("-")[-1]),\n    ) if root.exists() else []\n    return str(candidates[-1]) if candidates else None\n\n\ndef run_one(study, task_key, model_name, method, seed, run_mode, epochs, limits=None):\n    cfg = load_config()\n    if method not in METHODS: raise ValueError(method)\n    model_slug = model_name.replace("/", "__")\n    run_dir = ROOT / "results" / study / run_mode / task_key / model_slug / method / f"seed_{seed}"\n    metrics_path = run_dir / "final_metrics.json"\n    if metrics_path.exists():\n        try:\n            existing = json.loads(metrics_path.read_text(encoding="utf-8"))\n            if existing.get("status") == "COMPLETE":\n                return existing\n        except (json.JSONDecodeError, OSError):\n            pass\n    run_dir.mkdir(parents=True, exist_ok=True)\n    checkpoint_at_start = _latest_checkpoint(run_dir)\n    previous_status = {}\n    if (run_dir / "status.json").exists():\n        try: previous_status = json.loads((run_dir / "status.json").read_text(encoding="utf-8"))\n        except Exception: previous_status = {}\n    status = {\n        "status": "RUNNING", "started_at": previous_status.get("started_at", now_iso()),\n        "resumed_at": now_iso() if checkpoint_at_start else None,\n        "resume_count": int(previous_status.get("resume_count", 0)) + (1 if checkpoint_at_start else 0),\n        "resumed_from_checkpoint": checkpoint_at_start,\n        "study": study, "task": task_key, "model": model_name, "method": method,\n        "seed": seed, "run_mode": run_mode,\n    }\n    atomic_json(run_dir / "status.json", status)\n    append_event(run_dir / "events.jsonl", {"event": "RUN_STARTED", **status})\n    try:\n        set_seed(seed)\n        ds = load_task(task_key, run_mode, limits)\n        spec = TASKS[task_key]\n        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False if "bertweet" in model_name.lower() else True)\n        def tokenize(batch): return tokenizer(batch["text"], truncation=True, max_length=cfg["max_length"])\n        tokenized = ds.map(tokenize, batched=True, remove_columns=["sample_id", "text"])\n        model = build_model(model_name, spec.num_labels, method, cfg)\n        counts = parameter_counts(model)\n        atomic_json(run_dir / "run_config.json", {\n            **status, "epochs": epochs, "hyperparameters": cfg, "runtime": runtime_info(),\n            "model_commit": getattr(model.config, "_commit_hash", None), **counts,\n        })\n        callback = AtomicEpochCallback(run_dir / "epoch_metrics.csv")\n        trainer_kwargs = dict(\n            model=model, args=training_args(run_dir, method, seed, epochs, run_mode, cfg),\n            train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],\n            data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,\n            callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["early_stopping_patience"]), callback],\n        )\n        if "processing_class" in inspect.signature(Trainer.__init__).parameters: trainer_kwargs["processing_class"] = tokenizer\n        else: trainer_kwargs["tokenizer"] = tokenizer\n        trainer = Trainer(**trainer_kwargs)\n        checkpoint = _latest_checkpoint(run_dir)\n        started = time.perf_counter()\n        trainer.train(resume_from_checkpoint=checkpoint)\n        train_seconds = time.perf_counter() - started\n        test = trainer.predict(tokenized["test"], metric_key_prefix="test")\n        predictions = np.argmax(test.predictions, axis=-1)\n        pd.DataFrame({"sample_id": ds["test"]["sample_id"], "label": test.label_ids, "prediction": predictions}).to_csv(run_dir / "predictions.csv", index=False, encoding="utf-8-sig")\n        history = pd.DataFrame(trainer.state.log_history)\n        history.to_csv(run_dir / "trainer_history.csv", index=False, encoding="utf-8-sig")\n        result = {\n            "status": "COMPLETE", "completed_at": now_iso(), "study": study, "task": task_key,\n            "model": model_name, "method": method, "seed": seed, "run_mode": run_mode,\n            "train_rows": len(ds["train"]), "validation_rows": len(ds["validation"]), "test_rows": len(ds["test"]),\n            "epochs_requested": 1 if run_mode == "SMOKE" else epochs, "learning_rate": cfg["learning_rates"][method],\n            "train_seconds": train_seconds, **counts, **test.metrics, "runtime": runtime_info(),\n            "best_checkpoint": trainer.state.best_model_checkpoint,\n        }\n        atomic_json(metrics_path, result)\n        atomic_json(run_dir / "status.json", result)\n        append_event(run_dir / "events.jsonl", {"event": "RUN_COMPLETE", "test_macro_f1": result.get("test_macro_f1")})\n        if not cfg["keep_best_checkpoint"]:\n            shutil.rmtree(run_dir / "checkpoints", ignore_errors=True)\n        del trainer, model\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n        return result\n    except Exception as exc:\n        failed = {**status, "status": "FAILED", "failed_at": now_iso(), "error_type": type(exc).__name__, "error": str(exc)}\n        atomic_json(run_dir / "status.json", failed)\n        (run_dir / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")\n        append_event(run_dir / "events.jsonl", {"event": "RUN_FAILED", "error": str(exc)})\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n        raise\n\n\ndef build_jobs(study):\n    cfg = load_config(); section = cfg[study]\n    mode_limits = section.get("limits")\n    jobs = []\n    for task in section["tasks"]:\n        for model in section["models"]:\n            for method in cfg["methods"]:\n                for seed in cfg["seeds"]:\n                    jobs.append({"study": study, "task_key": task, "model_name": model, "method": method, "seed": seed, "epochs": section["epochs"], "limits": mode_limits})\n    return jobs\n\n\ndef run_study(study, run_mode="SMOKE", max_jobs=None, continue_on_error=None):\n    if run_mode not in {"SMOKE", "PAPER"}: raise ValueError("run_mode은 SMOKE 또는 PAPER만 가능합니다.")\n    precheck(require_cuda=True)\n    cfg = load_config(); jobs = build_jobs(study)\n    if max_jobs is not None: jobs = jobs[:max_jobs]\n    continue_on_error = cfg["continue_on_error"] if continue_on_error is None else continue_on_error\n    progress_path = ROOT / "results" / study / run_mode / "progress.json"\n    events_path = ROOT / "results" / study / run_mode / "events.jsonl"\n    completed, failed = 0, 0\n    results = []\n    for index, job in enumerate(jobs, 1):\n        current = {"study": study, "run_mode": run_mode, "total": len(jobs), "index": index, "completed": completed, "failed": failed, "current": job, "updated_at": now_iso()}\n        atomic_json(progress_path, current); append_event(events_path, {"event": "JOB_DISPATCH", "index": index, **job})\n        try:\n            result = run_one(run_mode=run_mode, **job)\n            results.append(result); completed += 1\n        except Exception:\n            failed += 1\n            if not continue_on_error:\n                atomic_json(progress_path, {**current, "status": "STOPPED_ON_ERROR", "failed": failed, "updated_at": now_iso()})\n                raise\n        atomic_json(progress_path, {**current, "status": "RUNNING", "completed": completed, "failed": failed, "updated_at": now_iso()})\n    final = {"study": study, "run_mode": run_mode, "status": "COMPLETE" if failed == 0 else "COMPLETE_WITH_ERRORS", "total": len(jobs), "completed": completed, "failed": failed, "updated_at": now_iso()}\n    atomic_json(progress_path, final); append_event(events_path, {"event": "STUDY_FINISHED", **final})\n    return pd.DataFrame(results)\n\n\ndef aggregate(run_mode="PAPER"):\n    rows = []\n    for path in (ROOT / "results").glob(f"study*/{run_mode}/**/final_metrics.json"):\n        row = json.loads(path.read_text(encoding="utf-8")); row["source_file"] = str(path.relative_to(ROOT)); rows.append(row)\n    frame = pd.DataFrame(rows)\n    out = ROOT / "results" / "aggregate"; out.mkdir(parents=True, exist_ok=True)\n    frame.to_csv(out / f"all_runs_{run_mode.lower()}.csv", index=False, encoding="utf-8-sig")\n    if not frame.empty:\n        summary = frame.groupby(["study", "task", "model", "method"], as_index=False).agg(\n            runs=("seed", "count"), macro_f1_mean=("test_macro_f1", "mean"), macro_f1_std=("test_macro_f1", "std"),\n            accuracy_mean=("test_accuracy", "mean"), train_seconds_mean=("train_seconds", "mean"),\n            trainable_ratio_mean=("trainable_parameter_ratio", "mean"),\n        )\n        summary.to_csv(out / f"summary_{run_mode.lower()}.csv", index=False, encoding="utf-8-sig")\n    return frame\n'
(DRIVE_ROOT / 'src' / 'suite.py').write_text(SUITE_SOURCE, encoding='utf-8')
print('suite.py written:', len(SUITE_SOURCE), 'chars')


In [ ]:
CONFIG_SOURCE = '{\n  "methods": ["full_ft", "lora", "adapter", "ia3", "bitfit"],\n  "seeds": [42, 52, 62, 72, 82],\n  "max_length": 128,\n  "batch_size": 16,\n  "eval_batch_size": 32,\n  "gradient_accumulation_steps": 4,\n  "precision": "fp16",\n  "weight_decay": 0.01,\n  "warmup_ratio": 0.06,\n  "early_stopping_patience": 2,\n  "dataloader_num_workers": 2,\n  "learning_rates": {\n    "full_ft": 0.00002,\n    "lora": 0.0001,\n    "adapter": 0.0001,\n    "ia3": 0.0005,\n    "bitfit": 0.0001\n  },\n  "lora": {"r": 8, "alpha": 16, "dropout": 0.05},\n  "adapter": {"bottleneck": 64, "dropout": 0.0},\n  "study1": {"epochs": 3, "models": ["vinai/bertweet-base"], "tasks": ["measuring_hate_speech"]},\n  "study2": {\n    "epochs": 2,\n    "models": ["vinai/bertweet-base", "FacebookAI/roberta-base"],\n    "tasks": ["tweet_sentiment", "finance_sentiment", "movie_reviews", "product_reviews", "tweet_emotion", "tweet_hate", "tweet_offensive", "tweet_irony", "news_topic"],\n    "limits": null\n  },\n  "study3": {"epochs": 5, "models": ["klue/roberta-base"], "tasks": ["news_ynat", "movie_nsmc", "comment_kmhas_binary"]},\n  "smoke_limits": {"train": 128, "validation": 64, "test": 64},\n  "keep_best_checkpoint": true,\n  "continue_on_error": false\n}\n'
(DRIVE_ROOT / 'config' / 'experiment_config.json').write_text(CONFIG_SOURCE, encoding='utf-8')
print(CONFIG_SOURCE)


## 3. A100 및 프로토콜 사전점검

GPU 이름이 A100이 아니면 실행을 중단합니다. 로컬과 동일하게 FP16을 사용하며 A100이라고 BF16이나 batch size를 변경하지 않습니다.


In [ ]:
sys.path.insert(0, str(DRIVE_ROOT))
from src.suite import precheck, load_config, build_jobs, run_study, aggregate

info = precheck(require_cuda=True)
display(info)
if 'A100' not in info['gpu'].upper():
    raise RuntimeError(f"A100 런타임이 아닙니다: {info['gpu']}")

cfg = load_config()
assert cfg['methods'] == ['full_ft', 'lora', 'adapter', 'ia3', 'bitfit']
assert cfg['seeds'] == [42, 52, 62, 72, 82]
assert cfg['precision'] == 'fp16'
assert cfg['batch_size'] == 16
assert cfg['gradient_accumulation_steps'] == 4
assert cfg['study2']['limits'] is None
print('PROTOCOL CHECK PASS')
print('Study 1 jobs:', len(build_jobs('study1')))
print('Study 2 jobs:', len(build_jobs('study2')))
print('Study 3 jobs:', len(build_jobs('study3')))


## 4. Study 1 실행 — 25 Runs

BERTweet + Measuring Hate Speech, 5 methods × 5 seeds. 예상 A100 시간은 로컬보다 짧지만 Drive 저장시간에 따라 달라질 수 있습니다.


In [ ]:
STUDY = 'study1'
RUN_MODE = 'PAPER'
study1_result = run_study(STUDY, run_mode=RUN_MODE, max_jobs=None, continue_on_error=False)
display(study1_result.tail())


## 5. Study 3 실행 — 75 Runs

KLUE-RoBERTa + YNAT/NSMC/K-MHaS, 5 methods × 5 seeds.


In [ ]:
STUDY = 'study3'
RUN_MODE = 'PAPER'
study3_result = run_study(STUDY, run_mode=RUN_MODE, max_jobs=None, continue_on_error=False)
display(study3_result.tail())


## 6. Study 2 실행 — 450 Runs

9 English tasks × 2 models × 5 methods × 5 seeds. 원본 split 전체를 사용합니다.


In [ ]:
STUDY = 'study2'
RUN_MODE = 'PAPER'
study2_result = run_study(STUDY, run_mode=RUN_MODE, max_jobs=None, continue_on_error=False)
display(study2_result.tail())


## 7. 전체 결과 집계

세 Study가 완료된 뒤 실행합니다.


In [ ]:
all_runs = aggregate('PAPER')
display(all_runs.head())
print('completed result rows:', len(all_runs), '/ 550')
assert len(all_runs) == 550, '아직 완료되지 않은 run이 있습니다.'


## 8. 진행상태 확인

학습 셀이 중단된 뒤 또는 Study 사이에 실행할 수 있습니다.


In [ ]:
from collections import Counter

for study in ('study1', 'study3', 'study2'):
    root = DRIVE_ROOT / 'results' / study / 'PAPER'
    statuses = []
    for path in root.glob('**/status.json') if root.exists() else []:
        try:
            statuses.append(json.loads(path.read_text(encoding='utf-8')).get('status', 'UNKNOWN'))
        except Exception:
            statuses.append('UNREADABLE')
    print(study, dict(Counter(statuses)))
    progress = root / 'progress.json'
    if progress.exists():
        print(progress.read_text(encoding='utf-8'))


## 재개 방법

1. Colab 연결이 끊기면 1~3번 셀을 다시 실행합니다.
2. 중단된 Study의 실행 셀을 다시 실행합니다.
3. `COMPLETE` run은 자동으로 건너뜁니다.
4. 중단 run은 Google Drive에 저장된 마지막 정상 epoch checkpoint에서 재개합니다.
5. 오류 발생 시 해당 run 폴더의 `error.txt`와 `status.json`을 확인합니다.
